In [ ]:
import os
import requests
import time
from tqdm import tqdm
import pandas as pd

import project_dirs as pdir

# Desired local directory
output_dir = pdir.DATA_DIR
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# List of files to download
csv_Data = pd.read_csv(os.path.join(pdir.CSV_DIR, "CLWD_OneSlide.csv"))
file_names = csv_Data["WSI_ID"].to_list()

In [ ]:
base_url = "https://leelab.kmmu.edu.cn/PathologyRepository/api/download"

# Number of retries for failed/incomplete downloads
max_retries = 5

# Chunk size: 1 MB
chunk_size = 1024 * 1024

# Create output directory
os.makedirs(output_dir, exist_ok=True)


for file_name in file_names:

    file_name = f"{file_name}.svs"

    output_path = os.path.join(output_dir, file_name)
    temp_path = output_path + ".part"

    # ---------------------------------------------------------
    # Skip if the final file already exists
    # ---------------------------------------------------------
    if os.path.exists(output_path):
        print(f"{file_name} already downloaded")
        continue

    url = f"{base_url}?file={file_name}"

    print(f"\nDownloading: {file_name}")

    success = False

    # ---------------------------------------------------------
    # Retry loop
    # ---------------------------------------------------------
    for attempt in range(1, max_retries + 1):

        try:

            print(f"Attempt {attempt}/{max_retries}")

            # Remove previous incomplete download
            if os.path.exists(temp_path):
                os.remove(temp_path)

            # -------------------------------------------------
            # Send request
            # -------------------------------------------------
            with requests.get(
                url,
                stream=True,
                timeout=(30, 300)
            ) as response:

                response.raise_for_status()

                # Expected file size from server
                content_length = response.headers.get("Content-Length")

                if content_length is not None:
                    expected_size = int(content_length)
                    print(
                        f"Expected size: "
                        f"{expected_size / (1024**3):.2f} GB"
                    )
                else:
                    expected_size = None
                    print("Warning: Server did not provide Content-Length")

                downloaded_size = 0

                # -------------------------------------------------
                # Write to temporary file
                # -------------------------------------------------
                with open(temp_path, "wb") as f:

                    with tqdm(
                        total=expected_size,
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        desc=file_name
                    ) as pbar:

                        for chunk in response.iter_content(
                            chunk_size=chunk_size
                        ):

                            if not chunk:
                                continue

                            f.write(chunk)

                            downloaded_size += len(chunk)
                            pbar.update(len(chunk))

                    # Make sure all data is flushed to disk
                    f.flush()
                    os.fsync(f.fileno())

            # -------------------------------------------------
            # Verify downloaded size
            # -------------------------------------------------
            print(
                f"Downloaded size: "
                f"{downloaded_size / (1024**3):.2f} GB"
            )

            if expected_size is not None:

                if downloaded_size != expected_size:

                    raise IOError(
                        f"Incomplete download! "
                        f"Expected {expected_size:,} bytes, "
                        f"but received {downloaded_size:,} bytes."
                    )

            # -------------------------------------------------
            # Verify local file size as an additional check
            # -------------------------------------------------
            actual_size = os.path.getsize(temp_path)

            if actual_size != downloaded_size:

                raise IOError(
                    f"File size mismatch! "
                    f"Downloaded {downloaded_size:,} bytes, "
                    f"but local file contains {actual_size:,} bytes."
                )

            # -------------------------------------------------
            # Download completed successfully
            # Rename .part -> .svs
            # -------------------------------------------------
            os.replace(temp_path, output_path)

            print(f"✓ Download complete: {output_path}")

            success = True
            break

        except Exception as e:

            print(f"✗ Download failed: {e}")

            # Delete incomplete file
            if os.path.exists(temp_path):
                os.remove(temp_path)

            if attempt < max_retries:

                print("Retrying in 5 seconds...")
                time.sleep(5)

            else:

                print(
                    f"✗ Failed to download {file_name} "
                    f"after {max_retries} attempts."
                )

    # ---------------------------------------------------------
    # Final status
    # ---------------------------------------------------------
    if not success:
        print(f"WARNING: {file_name} was NOT downloaded.")